# 04: Advanced Retrieval — Hybrid Search, Filters & Reranking

## Series Overview

Welcome to Part 4 of the RAG series for financial documents. We've built the foundation (embeddings, vector storage, basic retrieval). Now we tackle **production-grade advanced retrieval** that handles real-world financial documents with precision.

### What We Build

A multi-stage retrieval pipeline that combines:
- **Hybrid search**: Dense + sparse fusion for both semantic and exact matches
- **Metadata filtering**: Pre-filter by date, document type, client
- **Cross-encoder reranking**: Re-order top-K with a more powerful model

### Architecture Evolution

```
Naive RAG (Week 1):
  Query → Dense embedding search → Top 5 → GPT → Answer

Advanced RAG (This week):
  Query → [Dense + Sparse] → RRF fusion → Metadata filter → Reranker → Top 3 → GPT → Answer
```

Why does this matter? Real financial documents have:
- **Exact terminology**: "EBITDA margin", "force majeure", "Section 4.2(b)"
- **Numerical precision**: "18.3%" vs "18.2%" matters
- **Regulatory constraints**: Filter by year, document type, classification

Let's build it step-by-step.

## Setup & Imports

In [ ]:
import os
import sys
import json
from typing import Optional, List, Dict, Any
import numpy as np

# Custom imports (would be from src in a real project)
# from src.from_scratch.retrieval.dense import DenseRetriever
# from src.from_scratch.retrieval.sparse import BM25Retriever, build_bm25_retriever
# from src.from_scratch.retrieval.hybrid import HybridRetriever, hybrid_search
# from src.from_scratch.retrieval.metadata_filter import MetadataFilter, to_qdrant_filter, public_only, by_year, by_doc_type
# from src.from_scratch.retrieval.reranker import CrossEncoderReranker, rerank_chunks

print("✓ Imports ready")
print(f"  Python: {sys.version.split()[0]}")
print(f"  NumPy: {np.__version__}")

## Why Dense Embedding Search Alone Fails

Dense embeddings excel at **semantic similarity** but struggle with **exact terminology** and **precision** in finance.

### Two Key Failure Modes

**1. Vocabulary Mismatch**

Query: *"EBITDA margin Q3"*

Dense embedding model sees this and tries to find semantically similar text. But:
- Acronyms like "EBITDA" may not be well-distinguished in embeddings (trained on general text)
- "Q3" gets embedded the same as "Q2" or "quarter 3"
- Result: Your embedding might find "profitability improved" (semantic match) but miss "EBITDA margin improved 26%" (exact match)

**2. Precision Loss**

Query: *"revenue grew 18.3 percent"*

Dense search finds documents mentioning "revenue grew" (good!), but:
- The exact percentage "18.3%" is lost in embeddings
- You get chunks about "revenue grew 12%" and "revenue grew 25%"
- User actually needed the 18.3% figure for compliance reporting

### The Solution: Hybrid Search

Combine:
- **Dense (bi-encoder)**: Fast, semantic, good at "what does this mean?"
- **Sparse (BM25)**: Exact keywords, great for "does this chunk contain this exact word?"

Then **merge rankings** with Reciprocal Rank Fusion (RRF) to get the best of both worlds.

## BM25 Sparse Retrieval Explained

### What is BM25?

**BM25** ("Best Match 25") is the industry-standard sparse (keyword-based) retrieval algorithm. It's a probabilistic model that scores documents based on:

**TF-IDF with saturation:**
- **TF** (Term Frequency): How often does the query term appear in this document?
- **IDF** (Inverse Document Frequency): Is this term rare (high IDF) or common (low IDF)? "EBITDA" is rarer → higher weight.
- **Saturation**: Diminishing returns — 10 mentions of "revenue" isn't 10x better than 1 mention.

**Key parameters:**
- `k1 = 1.5`: Controls term frequency saturation (higher = more impact of repeated terms)
- `b = 0.75`: Controls length normalization (0 = ignore doc length, 1 = penalize long docs heavily)

### Why BM25 for Finance?

- ✓ Exact acronyms: "EBITDA", "GAAP", "NYSE"
- ✓ Precise numbers: "18.3%" is not conflated with "18.2%"
- ✓ Legal terms: "force majeure", "indemnification", "Section 4.2(b)"
- ✓ Ultra-fast: No neural network, just inverted index lookup
- ✗ No semantics: "revenue growth" ≠ "sales increase" (different words = 0 match)

In [ ]:
# Mock chunk and retriever classes for demo purposes
class MockChunk:
    """Simple chunk object with text and metadata."""
    def __init__(self, text, metadata=None):
        self.text = text
        self.metadata = metadata or {}

class RetrievedResult:
    """Result object returned by retrievers."""
    def __init__(self, text, score, rank, metadata=None):
        self.text = text
        self.score = score
        self.rank = rank
        self.metadata = metadata or {}

# Create sample financial document chunks
chunks = [
    MockChunk(
        "Revenue grew 18.3% year-over-year to $15.2B in Q3 2023",
        {"doc_type": "annual_report", "year": 2023, "page": 5}
    ),
    MockChunk(
        "Operating expenses increased 12% driven by headcount growth and infrastructure investment",
        {"doc_type": "annual_report", "year": 2023, "page": 6}
    ),
    MockChunk(
        "EBITDA margin improved from 22% to 26.4% in fiscal 2023, demonstrating operational leverage",
        {"doc_type": "annual_report", "year": 2023, "page": 8}
    ),
    MockChunk(
        "The company entered into a force majeure clause under Section 4.2(b) of the supplier agreement",
        {"doc_type": "contract", "year": 2022, "page": 12}
    ),
    MockChunk(
        "Net income reached $3.1B representing a net margin of 20.4%, up from 18.9% prior year",
        {"doc_type": "annual_report", "year": 2023, "page": 9}
    ),
    MockChunk(
        "Cloud services division drove 65% of total revenue growth with bookings up 42% year-over-year",
        {"doc_type": "annual_report", "year": 2023, "page": 7}
    ),
]

print("Sample corpus (6 chunks from financial documents):")
print()
for i, chunk in enumerate(chunks, 1):
    print(f"[{i}] {chunk.text}")
    print(f"    Metadata: {chunk.metadata}")
    print()

In [ ]:
# Simplified BM25 implementation for demo
from collections import defaultdict
import math

class SimpleBM25Retriever:
    """Simplified BM25 for educational purposes."""
    def __init__(self, chunks, k1=1.5, b=0.75):
        self.chunks = chunks
        self.k1 = k1
        self.b = b
        self._build_index()
    
    def _build_index(self):
        """Build inverted index and IDF values."""
        self.doc_freqs = defaultdict(int)
        self.index = defaultdict(list)
        self.doc_lengths = []
        
        for doc_id, chunk in enumerate(self.chunks):
            words = chunk.text.lower().split()
            self.doc_lengths.append(len(words))
            seen = set()
            
            for word in words:
                self.index[word].append((doc_id, words.count(word)))
                if word not in seen:
                    self.doc_freqs[word] += 1
                    seen.add(word)
        
        self.avgdl = sum(self.doc_lengths) / len(self.doc_lengths)
        self.num_docs = len(self.chunks)
    
    def search(self, query, top_k=3):
        """Search for query, return top_k results."""
        query_words = query.lower().split()
        scores = defaultdict(float)
        
        for word in query_words:
            if word not in self.index:
                continue
            
            idf = math.log((self.num_docs - self.doc_freqs[word] + 0.5) / (self.doc_freqs[word] + 0.5))
            
            for doc_id, freq in self.index[word]:
                doc_len = self.doc_lengths[doc_id]
                numerator = freq * (self.k1 + 1)
                denominator = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                scores[doc_id] += idf * (numerator / denominator)
        
        # Sort and return
        ranked = sorted(scores.items(), key=lambda x: -x[1])
        results = []
        for rank, (doc_id, score) in enumerate(ranked[:top_k], 1):
            results.append(
                RetrievedResult(
                    text=self.chunks[doc_id].text,
                    score=score,
                    rank=rank,
                    metadata=self.chunks[doc_id].metadata
                )
            )
        return results

# Initialize BM25 retriever
retriever_bm25 = SimpleBM25Retriever(chunks)

# Search for EBITDA-related content
print("BM25 Search Results for 'EBITDA margin 2023':")
print()
results = retriever_bm25.search("EBITDA margin 2023", top_k=3)
for r in results:
    print(f"Rank {r.rank} | BM25 Score {r.score:.3f}")
    print(f"  Text: {r.text}")
    print(f"  Metadata: {r.metadata}")
    print()

# Another search: exact legal term
print("\nBM25 Search Results for 'force majeure clause':")
print()
results = retriever_bm25.search("force majeure clause", top_k=2)
for r in results:
    print(f"Rank {r.rank} | BM25 Score {r.score:.3f}")
    print(f"  Text: {r.text}")
    print()

## The Vocabulary Gap: Dense vs Sparse

Let's see concretely how dense and sparse retrievers complement each other:

| Query | What Dense Finds | What BM25 Finds | Winner |
|-------|------------------|-----------------|--------|
| "EBITDA margin" | "profitability improved" (semantic, but wrong acronym) | "EBITDA margin improved 26.4%" (exact!) | **BM25** |
| "profit growth" | "revenue grew 18.3%" (semantic synonym) | Nothing (different words) | **Dense** |
| "force majeure" | Unrelated legal clauses (no semantic signal) | "force majeure clause under Section 4.2(b)" (exact match!) | **BM25** |
| "company bookings increase" | "Cloud services drove 65% of revenue growth" (semantic) | "bookings up 42%" (exact) | **Both!** |

### Conclusion

**No single retriever is perfect.**
- Dense = "what does this mean?"
- Sparse = "does this contain these exact words?"

Hybrid search fuses both rankings to give you the best of both worlds.

## Hybrid Search with Reciprocal Rank Fusion (RRF)

### The Problem with Simple Score Merging

Dense embeddings return scores like `[0.91, 0.87, 0.72]` (cosine similarity, 0-1 range).
BM25 returns scores like `[8.4, 7.1, 3.2]` (unbounded, depends on query and corpus).

**Can't add them directly** — they're on different scales!

### Reciprocal Rank Fusion (RRF)

RRF converts scores to **ranks**, then fuses ranks using a principled formula:

$$\text{RRF Score} = \sum_i \frac{1}{k + \text{rank}_i}$$

Where:
- `k = 60` (standard constant, prevents division by zero)
- `rank_i` = rank in retriever i (1st place, 2nd place, etc.)

**Intuition:**
- Top rank (1st) contributes `1/(60+1) = 0.0164`
- 2nd rank contributes `1/(60+2) = 0.0161`
- 10th rank contributes `1/(60+10) = 0.0125`

**Property:** A document ranked #1 by BOTH retrievers beats a document ranked #1 by one and #10 by the other.

This is the **Spearman rank correlation** formula — it's been used in information retrieval for decades.

In [ ]:
# Mock dense retriever for demo (in production: uses embeddings + vector DB)
class MockDenseRetriever:
    """Mock dense retriever returning pre-defined results to illustrate concept."""
    def search(self, query, top_k=5):
        """Return mock dense search results."""
        return [
            RetrievedResult(
                "Revenue grew 18.3% year-over-year to $15.2B in Q3 2023",
                0.91, 1, {"page": 5}
            ),
            RetrievedResult(
                "Cloud services drove 65% of total revenue growth with bookings up 42% year-over-year",
                0.87, 2, {"page": 7}
            ),
            RetrievedResult(
                "Net income reached $3.1B representing a net margin of 20.4%, up from 18.9% prior year",
                0.72, 3, {"page": 9}
            ),
        ]

class MockSparseRetriever:
    """Mock sparse (BM25) retriever returning pre-defined results."""
    def search(self, query, top_k=5):
        """Return mock BM25 search results."""
        return [
            RetrievedResult(
                "EBITDA margin improved from 22% to 26.4% in fiscal 2023, demonstrating operational leverage",
                8.4, 1, {"page": 8}
            ),
            RetrievedResult(
                "Revenue grew 18.3% year-over-year to $15.2B in Q3 2023",
                7.1, 2, {"page": 5}
            ),
            RetrievedResult(
                "Operating expenses increased 12% driven by headcount growth and infrastructure investment",
                3.2, 3, {"page": 6}
            ),
        ]

# Perform RRF fusion manually to show the math
k = 60
query = "revenue growth profitability"

dense_results = MockDenseRetriever().search(query)
sparse_results = MockSparseRetriever().search(query)

print(f"Query: '{query}'")
print(f"\nDense Retriever Results (cosine similarity):")
for r in dense_results:
    print(f"  [{r.rank}] score={r.score:.2f} | {r.text[:60]}...")

print(f"\nSparse (BM25) Retriever Results:")
for r in sparse_results:
    print(f"  [{r.rank}] score={r.score:.2f} | {r.text[:60]}...")

# Build a mapping from text to ranks
dense_ranks = {r.text[:50]: r.rank for r in dense_results}
sparse_ranks = {r.text[:50]: r.rank for r in sparse_results}

# Compute RRF scores
print(f"\n" + "="*90)
print(f"RRF Fusion (k={k})")
print("="*90)
print(f"{'Document':<52} {'Dense':>7} {'Sparse':>7} {'RRF Score':>12}")
print("-"*90)

all_texts = set(list(dense_ranks.keys()) + list(sparse_ranks.keys()))
fused_scores = []

for text in all_texts:
    d_rank = dense_ranks.get(text, None)
    s_rank = sparse_ranks.get(text, None)
    
    rrf = (1/(k + d_rank) if d_rank else 0) + (1/(k + s_rank) if s_rank else 0)
    fused_scores.append((text, d_rank, s_rank, rrf))

# Sort by RRF score (descending)
for text, d_rank, s_rank, rrf in sorted(fused_scores, key=lambda x: -x[3]):
    d_str = str(d_rank) if d_rank else "-"
    s_str = str(s_rank) if s_rank else "-"
    print(f"{text:<52} {d_str:>7} {s_str:>7} {rrf:>12.5f}")

print()
print("Key insights:")
print(f"  • 'Revenue grew...' appears in both (rank 1 & 2) → top fusion score")
print(f"  • 'EBITDA margin...' only in sparse (rank 1) → still scores well")
print(f"  • This is why RRF works: combines complementary signals")

## Metadata Filtering: Scope Reduction

### Why Filter?

Real-world document collections are **huge**:
- 100K+ chunks across decades
- Multi-tenant (different clients, subject to confidentiality)
- Different document types (annual reports, contracts, emails, research)

**Querying all 100K chunks is slow and irrelevant.** 

Metadata filters **reduce the search space** before expensive retrieval:
```
All chunks (100K) 
  → Filter: doc_type=annual_report AND year>=2022 (5K chunks)
  → Retrieve top-10 from 5K (fast, focused)
  → Rerank → return top-3
```

### Common Filters for Finance

- **doc_type**: `"annual_report"`, `"contract"`, `"press_release"`, `"sec_filing"`
- **year**: `2023`, `2022`, etc. (or range: `year >= 2022`)
- **client**: `"Client A"`, `"Client B"` (multi-tenant)
- **classification**: `"public"`, `"confidential"`, `"restricted"`
- **sector**: `"healthcare"`, `"finance"`, `"technology"`

In production, these become **Qdrant filters** (run in the vector DB for speed).

In [ ]:
# Metadata filtering implementation

class MetadataFilter:
    """Simple metadata filter."""
    def __init__(self, **kwargs):
        self.conditions = kwargs
    
    def matches(self, metadata):
        """Check if metadata dict matches all conditions."""
        for key, value in self.conditions.items():
            if key not in metadata or metadata[key] != value:
                return False
        return True

def apply_filter_to_list(chunks, filt):
    """Filter list of chunks by metadata."""
    return [c for c in chunks if filt.matches(c.metadata)]

# Factory functions for common filters
def public_only():
    return MetadataFilter(classification="public")

def by_year(year):
    return MetadataFilter(year=year)

def by_doc_type(doc_type):
    return MetadataFilter(doc_type=doc_type)

def to_qdrant_filter(filt):
    """Convert to Qdrant filter format (for vector DB)."""
    # In production, this would generate Qdrant HasField/Match filters
    return {"metadata_filter": filt.conditions}

# Demo filtering
print(f"Total chunks in corpus: {len(chunks)}")
print()

# Filter 1: Annual reports only
filt1 = MetadataFilter(doc_type="annual_report")
filtered1 = apply_filter_to_list(chunks, filt1)
print(f"Filter: doc_type='annual_report'")
print(f"  Result: {len(filtered1)} chunks")
for c in filtered1:
    print(f"    • {c.text[:60]}...")
print()

# Filter 2: 2023 documents only
filt2 = MetadataFilter(year=2023)
filtered2 = apply_filter_to_list(chunks, filt2)
print(f"Filter: year=2023")
print(f"  Result: {len(filtered2)} chunks")
for c in filtered2:
    print(f"    • {c.text[:60]}...")
print()

# Filter 3: Combination (AND logic)
filt3 = MetadataFilter(doc_type="annual_report", year=2023)
filtered3 = apply_filter_to_list(chunks, filt3)
print(f"Filter: doc_type='annual_report' AND year=2023")
print(f"  Result: {len(filtered3)} chunks")
for c in filtered3:
    print(f"    • {c.text[:60]}...")
print()

# Show Qdrant filter format
print("Qdrant Filter Format (for production):")
qdrant_filt = to_qdrant_filter(filt3)
print(f"  {json.dumps(qdrant_filt, indent=2)}")
print()

# Show factory functions
print("Factory Functions for Common Filters:")
print(f"  public_only() -> {public_only().conditions}")
print(f"  by_year(2023) -> {by_year(2023).conditions}")
print(f"  by_doc_type('contract') -> {by_doc_type('contract').conditions}")

## Cross-Encoder Reranking

### Bi-Encoder vs Cross-Encoder

**Bi-Encoder (what we use for retrieval):**
```
Query → Encoder → Embedding Q
Document → Encoder → Embedding D
Score = cosine_similarity(Q, D)
```
- ✓ Fast: Encode query once, docs in batches
- ✗ Limited: Separate representations miss interaction signals
- Use for: Retrieving top-50 candidates

**Cross-Encoder (what we use for reranking):**
```
[Query, Document] → Encoder → Relevance Score (0-1)
```
- ✓ Powerful: Directly models (query, doc) interaction
- ✗ Slow: Must encode each (query, doc) pair separately
- Use for: Reranking top-10 → top-3

### The Retrieval Funnel

```
100K docs
   ↓ (metadata filter)
5K docs
   ↓ (dense + sparse retrieval)
20 docs
   ↓ (cross-encoder reranking) ← EXPENSIVE, but worth it for precision
3 docs → Send to LLM
```

### Why It Works for Finance

Query: *"What was the profitability improvement?"*

**Dense ranking:**
1. "Revenue grew 18.3% year-over-year" (score 0.91) — about growth, but not profitability
2. "Cloud services drove 65% of growth" (score 0.87) — about growth segment
3. "EBITDA margin improved to 26.4%" (score 0.72) — **actually about profitability!**

**Cross-encoder reranking:**
- Scores how well (query, document) pair matches
- "Profitability improvement" + "EBITDA margin improved" = strong match (0.94)
- "Profitability improvement" + "Revenue grew" = weak match (0.71)

**Result after reranking:**
1. "EBITDA margin improved to 26.4%" (score 0.94) — jumped from #3 to #1!

This is **query-aware re-ordering** — different query → different ranking.

In [ ]:
# Cross-encoder reranking demo
# Note: In production, this loads a real cross-encoder model (e.g., MiniLM)
# For this demo, we use pre-computed scores to show the concept

class CrossEncoderReranker:
    """
    Cross-encoder reranker.
    In production: loads cross-encoder model, encodes (query, doc) pairs.
    For demo: uses pre-computed relevance scores.
    """
    def rerank(self, query, candidates, top_n=3):
        """
        Rerank candidates by cross-encoder score.
        Returns top_n re-ranked results.
        """
        # In production: would call model
        # For demo, return candidates sorted by relevance
        return sorted(candidates, key=lambda x: -x['ce_score'])[:top_n]

# Initial retrieval results (from dense + sparse + RRF)
initial_results = [
    {"rank": 1, "text": "Revenue grew 18.3% year-over-year to $15.2B",       "dense_score": 0.91},
    {"rank": 2, "text": "Cloud services drove 65% of total revenue growth", "dense_score": 0.87},
    {"rank": 3, "text": "EBITDA margin improved from 22% to 26.4% in 2023", "dense_score": 0.72},
    {"rank": 4, "text": "Operating expenses increased 12% driven by growth",    "dense_score": 0.68},
    {"rank": 5, "text": "Net income reached $3.1B, net margin 20.4%",    "dense_score": 0.61},
]

query = "What was the profitability improvement?"

# Add pre-computed cross-encoder scores
# Higher = more relevant to THIS specific query
ce_scores = {
    "Revenue grew 18.3%": 0.71,
    "Cloud services drove 65%": 0.68,
    "EBITDA margin improved": 0.94,  # Jumps up!
    "Operating expenses increased": 0.45,
    "Net income reached": 0.89,  # Also jumps up (profitability!)
}

for result in initial_results:
    result['ce_score'] = ce_scores.get(result['text'].split()[0:4][-1], 0)
    for key in ce_scores:
        if key in result['text']:
            result['ce_score'] = ce_scores[key]
            break

# For simplicity, assign scores directly
initial_results[0]['ce_score'] = 0.71
initial_results[1]['ce_score'] = 0.68
initial_results[2]['ce_score'] = 0.94
initial_results[3]['ce_score'] = 0.45
initial_results[4]['ce_score'] = 0.89

print(f"Query: '{query}'")
print()

print("BEFORE Cross-Encoder Reranking (Dense scores):")
print(f"{'Rank':<6} {'Score':<8} {'Text':<50}")
print("-"*70)
for r in initial_results[:3]:
    print(f"{r['rank']:<6} {r['dense_score']:<8.2f} {r['text']:<50}")

print()
print("AFTER Cross-Encoder Reranking (Relevance scores):")
print(f"{'Rank':<6} {'Score':<8} {'Text':<50} {'Change':<12}")
print("-"*80)

reranked = CrossEncoderReranker().rerank(query, initial_results, top_n=3)
for new_rank, result in enumerate(reranked, 1):
    old_rank = result['rank']
    if new_rank < old_rank:
        change = f"↑ #{old_rank}→#{new_rank}"
    elif new_rank > old_rank:
        change = f"↓ #{old_rank}→#{new_rank}"
    else:
        change = f"→ stayed #{old_rank}"
    
    print(f"{new_rank:<6} {result['ce_score']:<8.2f} {result['text']:<50} {change:<12}")

print()
print("✓ Key observation:")
print("  • 'EBITDA margin improved' jumped from rank #3 → #1")
print("  • 'Net income' jumped from rank #5 → #2")
print("  • Both directly answer 'profitability improvement'")
print("  • Dense embedding missed this nuance (ranked them lower)")
print()
print("💡 To use the real reranker (no download needed for demo):")
print("   from sentence_transformers import CrossEncoder")
print("   model = CrossEncoder('cross-encoder/miniLM-L6-v2')  # ~80MB")
print("   scores = model.predict([(query, doc) for doc in candidates])")

## Full Advanced Retrieval Pipeline

Now let's put it all together: a complete, production-grade advanced retrieval function that orchestrates:

1. **Metadata pre-filtering** — scope reduction
2. **Dense + Sparse retrieval** — dual signal
3. **RRF fusion** — merge rankings
4. **Cross-encoder reranking** — precision top-N

This is the **golden path** for financial document RAG.

In [ ]:
def advanced_rag_pipeline(
    query: str,
    all_chunks: list,
    metadata_filter: Optional[MetadataFilter] = None,
    top_k_retrieve: int = 10,
    top_n_rerank: int = 3,
    verbose: bool = True,
):
    """
    Complete advanced RAG retrieval pipeline.
    
    Steps:
    1. Metadata pre-filter to reduce search space
    2. Sparse retrieval (BM25) — exact keywords
    3. Dense retrieval (embeddings + Qdrant) — semantic match
    4. RRF fusion — merge rankings
    5. Cross-encoder reranking — final precision top-N
    
    Args:
        query: User query
        all_chunks: All document chunks
        metadata_filter: Optional MetadataFilter for pre-filtering
        top_k_retrieve: Retrieve this many before reranking
        top_n_rerank: Return top this many after reranking
        verbose: Print diagnostics
    
    Returns:
        List of top_n_rerank reranked results
    """
    if verbose:
        print(f"Query: '{query}'")
        print(f"Total chunks available: {len(all_chunks)}")
    
    # Step 1: Metadata pre-filter
    if metadata_filter:
        search_chunks = apply_filter_to_list(all_chunks, metadata_filter)
        if verbose:
            print(f"After metadata filter: {len(search_chunks)} chunks")
            print(f"  Filter: {metadata_filter.conditions}")
    else:
        search_chunks = all_chunks
        if verbose:
            print("No metadata filter applied")
    
    # Step 2: Sparse retrieval (BM25)
    if verbose:
        print(f"\nStep 2: Sparse retrieval (BM25)...")
    bm25 = SimpleBM25Retriever(search_chunks)
    sparse_results = bm25.search(query, top_k=top_k_retrieve)
    if verbose:
        print(f"  Got {len(sparse_results)} results")
        for r in sparse_results[:2]:
            print(f"    BM25[{r.rank}] score={r.score:.2f} | {r.text[:55]}...")
    
    # Step 3: Dense retrieval
    if verbose:
        print(f"\nStep 3: Dense retrieval (would use Qdrant)...")
        print(f"  (Requires OpenAI API key + vector embeddings)")
        print(f"  In this demo, we use BM25 results only")
    
    # Step 4: RRF fusion (if we had both dense and sparse)
    if verbose:
        print(f"\nStep 4: RRF fusion...")
        print(f"  (Would merge dense + sparse rankings)")
    
    # For demo, we'll use sparse results directly
    candidates = sparse_results
    
    # Step 5: Cross-encoder reranking
    if verbose:
        print(f"\nStep 5: Cross-encoder reranking (top {len(sparse_results)} → top {top_n_rerank})...")
        print(f"  (Would use cross-encoder model)")
    
    # Return top_n (in demo, just truncate; in production, use CrossEncoderReranker)
    final_results = candidates[:top_n_rerank]
    
    if verbose:
        print(f"\nFinal Results (top {len(final_results)}):")
        for i, r in enumerate(final_results, 1):
            print(f"  [{i}] {r.text}")
            print(f"      Score: {r.score:.3f}, Metadata: {r.metadata}")
    
    return final_results

# Run the pipeline
print("="*80)
print("ADVANCED RAG PIPELINE DEMO")
print("="*80)
print()

results = advanced_rag_pipeline(
    query="What was the profitability in fiscal 2023?",
    all_chunks=chunks,
    metadata_filter=MetadataFilter(doc_type="annual_report", year=2023),
    top_k_retrieve=5,
    top_n_rerank=3,
    verbose=True,
)

print()
print("="*80)

## LangChain Equivalents

The from-scratch approach gives **transparency and control**. 

In production with LangChain, you'd use higher-level abstractions that do the same thing under the hood, but with additional benefits (streaming, tracing, debugging).

In [ ]:
print("LangChain Integration Examples")
print("="*70)
print()

print("Option 1: Ensemble Retriever (multiple retrievers)")
print("-"*70)
print()
print("from langchain.retrievers import EnsembleRetriever")
print()
print("ensemble_retriever = EnsembleRetriever(")
print("    retrievers=[bm25_retriever, dense_retriever],")
print("    weights=[0.5, 0.5],  # Equal weighting")
print(")")
print()
print("results = ensemble_retriever.get_relevant_documents(query)")
print()
print()

print("Option 2: Contextual Compression (reranking)")
print("-"*70)
print()
print("from langchain.retrievers import ContextualCompressionRetriever")
print("from langchain.retrievers.document_compressors import CrossEncoderReranker")
print()
print("compressor = CrossEncoderReranker(")
print("    model=HuggingFaceCrossEncoder('cross-encoder/miniLM-L6-v2')")
print(")")
print()
print("compression_retriever = ContextualCompressionRetriever(")
print("    base_retriever=ensemble_retriever,")
print("    base_compressor=compressor,")
print(")")
print()
print("results = compression_retriever.get_relevant_documents(query)")
print()
print()

print("Option 3: Full Pipeline (recommended for production)")
print("-"*70)
print()
print("# Create the full retrieval chain:")
print("# 1. Metadata filter")
print("# 2. Hybrid (BM25 + Dense)")
print("# 3. Rerank with cross-encoder")
print()
print("from src.langchain_impl.retrieval.full_pipeline import build_full_advanced_retriever")
print()
print("retriever = build_full_advanced_retriever(")
print("    vectorstore=qdrant_vectorstore,")
print("    sparse_chunks=chunks,")
print("    metadata_filter=MetadataFilter(doc_type='annual_report', year=2023),")
print("    top_k_retrieve=10,")
print("    top_n_rerank=3,")
print(")")
print()
print("# Use in a chain")
print("results = retriever.invoke({'query': 'What was profitability in 2023?'})")
print()
print()

print("Benefits of LangChain abstractions:")
print("-"*70)
print("✓ Streaming: results come as they're retrieved")
print("✓ Tracing: every step logged to LangSmith")
print("✓ Caching: embeddings cached to save API calls")
print("✓ Fallback: automatic retry on errors")
print("✓ Composition: easily chain multiple retrievers")
print()
print("Trade-off:")
print("✗ Less transparent: harder to debug individual steps")
print("✗ Magic overhead: sometimes behavior surprising")

## Summary: Advanced Retrieval Techniques

### Technique Comparison

| Technique | Solves Problem | Complexity | When to Use | Cost |
|-----------|---|---|---|---|
| **Dense only** | Semantic matching | Low | Simple queries, narrative documents | 💰 API calls |
| **BM25 only** | Exact keyword match | Low | Codes, numbers, legal terms, no synonyms | Free |
| **Hybrid RRF** | Both semantic + exact | Medium | **Production default** | 💰💰 |
| **Metadata filter** | Scope reduction | Low | Multi-tenant, time-bounded searches | Free |
| **Cross-encoder rerank** | Precision top-N | High | When top-3 quality is critical | 💰 model inference |
| **Full pipeline** | Everything combined | High | **Real-world financial RAG** | 💰💰💰 |

### Decision Tree

```
Do you have exact terminology?
├─ YES: Add BM25
├─ NO: Dense only is fine

Need to scope down large corpus?
├─ YES: Add metadata filtering
└─ NO: Skip filtering

Is precision in top-3 critical?
├─ YES: Add cross-encoder reranking
└─ NO: RRF fusion is enough

→ Start simple (dense), add techniques as needed
→ Benchmark each stage to measure improvement
```

### Key Takeaways

1. **Dense embedding alone is not enough** — it misses exact terminology and loses precision
2. **BM25 is essential** — exact matches for financial acronyms and numbers
3. **RRF fusion combines both** — proven method to merge diverse rankings
4. **Metadata filtering is free optimization** — reduces search space 10-100x
5. **Cross-encoder reranking adds cost but improves quality** — worth it for high-stakes queries
6. **Complexity → Accuracy trade-off** — measure with RAGAS metrics (Week 5)

### What's Next: Week 5

**RAG Evaluation — How do we know if our retriever is good?**

We'll learn:
- **RAGAS metrics**: Faithfulness, Answer Relevancy, Context Precision, Context Recall
- **Automatic evaluation** without manual labeling
- **Benchmarking** against baselines
- **Debugging** failure cases

This completes the retrieval foundation. Weeks 5+ are about:
- Evaluation (Week 5)
- Integration with LLMs (Week 6)
- Fine-tuning and domain adaptation (Week 7)
- Production deployment (Week 8)